In [1]:
from totalsegmentator.python_api import totalsegmentator
from pathlib import Path
import yaml
import time
import json

In [2]:
def get_organ_task (organ_yaml: str):
    with open(organ_yaml,"r") as f:
        organs = yaml.safe_load(f)
        
    organs_in_total = []
    organs_in_head = []
    all_organs = []
    tasks = set()
    
    for organ in organs["organs"]:
        all_organs.append(organ["source_class"]) #get all required organ 
        tasks.add(organ["task"])
        
        if (organ["task"]) == "total":
            organs_in_total.append(organ["source_class"]) #get organ in total (for roi_subset)
        else:
            organs_in_head.append(organ["source_class"]) #get organ in head_glands_cavities (for masks selection)
            
    return all_organs, organs_in_total, organs_in_head, tasks   
    #list of all organ, organ in total (for roi_subset), organ in head_glands_cavities (for masks selection)

In [3]:
def get_params (data_path: str, organ_yaml: str):
    data_folder = Path(data_path)
    input_folder = data_folder / "input"
    output_folder = data_folder / "transit_folder" 
    params = []
    _, roi_subset, _, tasks = get_organ_task(organ_yaml) # get tasks and roi_subset
    
    for subject_dir in sorted(input_folder.iterdir()): #get subjects in input folder Subject = quadra 1-48
        if subject_dir.is_dir():
            subject_img_dirs = sorted(subject_dir.glob("*.nii.gz"))
            
            if not subject_img_dirs:
                raise ValueError (f"WARNING!!! images not found in this dir: '{subject_dir}'") 
            else:

                #get the output dir for TotalSegmentator
                for subject_img_dir in subject_img_dirs: 
                    session_name = subject_img_dir.name.removesuffix("_CT-AC.nii.gz") # get session_name --> Session = test, retest
                    subject_mask_dir = output_folder / subject_dir.name / session_name # formulating output dirs for TotalSegmentator
                    subject_mask_dir.mkdir(parents=True, exist_ok=True) #creating folder based on formulated dirs
                    
                    #listing the parameter
                    for task in tasks:
                        param = {
                            "subject_id": subject_dir.name,
                            "session": session_name,
                            "task": task,
                            "input": subject_img_dir,
                            "output": subject_mask_dir,
                            "device": "gpu"
                        }
                        if task == "total":
                            param["roi_subset"] = roi_subset #add roi_subset if the task is "toal"
                        params.append(param)
    
    
    return params #list of dict{parameter} (subject, session, task, input dir, output dir, device)

parameters = get_params (data_path="/workspaces/segmentation/data", organ_yaml="/workspaces/segmentation/organs.yaml")

In [4]:
parameters

[{'subject_id': 'QUADRA_HC_021',
  'session': 'retest',
  'task': 'total',
  'input': PosixPath('/workspaces/segmentation/data/input/QUADRA_HC_021/retest_CT-AC.nii.gz'),
  'output': PosixPath('/workspaces/segmentation/data/transit_folder/QUADRA_HC_021/retest'),
  'device': 'gpu',
  'roi_subset': ['brain',
   'skull',
   'vertebrae_C1',
   'vertebrae_C4',
   'vertebrae_C7',
   'lung_upper_lobe_left',
   'lung_lower_lobe_left',
   'lung_upper_lobe_right',
   'lung_middle_lobe_right',
   'lung_lower_lobe_right',
   'heart',
   'esophagus',
   'vertebrae_T4',
   'vertebrae_T8',
   'vertebrae_T12',
   'rib_left_6',
   'rib_right_6',
   'liver',
   'kidney_left',
   'kidney_right',
   'pancreas',
   'duodenum',
   'small_bowel',
   'colon',
   'vertebrae_L1',
   'vertebrae_L3',
   'vertebrae_L5',
   'urinary_bladder',
   'prostate',
   'sacrum',
   'hip_left',
   'hip_right']},
 {'subject_id': 'QUADRA_HC_021',
  'session': 'retest',
  'task': 'head_glands_cavities',
  'input': PosixPath('/wo

In [7]:
def write_report_json (parameters):
    json_dir = "/workspaces/segmentation/data/output/segmentation_parameters.json"
    parameters_json = []
    for parameter in parameters:
        parameter_json = {
            "subject": parameter["subject_id"],
            "session": parameter["session"],
            "input": str(parameter["input"]),
            "output": str(parameter["output"]),
            "task": parameter["task"],
            "roi_subset": parameter.get("roi_subset", None),
            "device": parameter["device"]
        }
        parameters_json.append(parameter_json)

    with open(json_dir, "w", encoding="utf-8") as file:
        json.dump(parameters_json, file, indent=4)
    print (f"The json file for segmentation parameters is exported in '{json_dir}")

In [ ]:
def run_segmentation (parameters):
    start_time = time.perf_counter()
    for parameter in parameters:
        if __name__ == "__main__":
            totalsegmentator(
                input = parameter["input"],
                output = parameter["output"],
                task = parameter["task"],
                roi_subset = parameter.get("roi_subset", None),
                device = parameter["device"]
            )
    end_time = time.perf_counter()
    execution_time = end_time - start_time
    print(f"Segmentation time: {execution_time:.6f} seconds")
    write_report_json(parameters)
    


In [ ]:
# this is just simulation to see how the parameter looks like in the main function

start_time = time.perf_counter()
for parameter in parameters:
    print (
        "input = ", parameter["input"], "\n",
        "output = ", parameter["output"], "\n",
        "task = ", parameter["task"], "\n",
        "device = ", parameter["device"], "\n"
        "roi_subset = ", parameter.get("roi_subset", None)
    )
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Segmentation time: {execution_time:.6f} seconds")


these two cells bellow are for testing time difference between using roi_subset or not

In [ ]:
start_time = time.perf_counter()
for parameter in parameters:
    if __name__ == "__main__":
        totalsegmentator(
            input = parameter["input"],
            output = parameter["output"],
            task = parameter["task"],
            roi_subset = parameter.get("roi_subset", None),
            device = parameter["device"]
        )
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Segmentation time: {execution_time:.6f} seconds")


In [ ]:
start_time = time.perf_counter()
for parameter in parameters:
    if __name__ == "__main__":
        totalsegmentator(
            input = parameter["input"],
            output = parameter["output"],
            task = parameter["task"],
            device = parameter["device"],
        )
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Segmentation time: {execution_time:.6f} seconds")


Segmentation time: 463.452297 seconds
Segmentation time: 1521.024349 seconds